## 1. Create fictional employee and permission data

In [6]:
employees = [
    {"employee_id": "E001", "name": "Aarav", "role": "Finance Analyst"},
    {"employee_id": "E002", "name": "Meera", "role": "HR Executive"},
    {"employee_id": "E003", "name": "Rohan", "role": "IT Support"},
]

# Standard permissions expected for each role
role_permissions = {
    "Finance Analyst": ["financial_reports", "budget_system"],
    "HR Executive": ["employee_records", "payroll_system"],
    "IT Support": ["system_logs", "user_account_management"],
}

# Permissions each employee currently has
assigned_permissions = {
    "E001": ["financial_reports", "budget_system"],
    "E002": ["employee_records", "payroll_system", "financial_reports", "customer_database"],
    "E003": ["system_logs", "user_account_management"],
}

# Example approved exception. Remove this permission from the list to test an alert.
"""
approved_exceptions = {
    "E002": ["financial_reports"]
}
"""
print(f"Loaded {len(employees)} fictional employees.")

Loaded 3 fictional employees.


## 2. Detect access that needs review
The function checks assigned permissions against the employee's role permissions and approved exceptions. It returns a list of report records.

In [7]:
def detect_access_drift(employees, role_permissions, assigned_permissions, approved_exceptions):
    report = []

    for employee in employees:
        employee_id = employee["employee_id"]
        role = employee["role"]

        # Use an empty list if a role or employee has no matching entry.
        expected = role_permissions.get(role, [])
        assigned = assigned_permissions.get(employee_id, [])
        exceptions = approved_exceptions.get(employee_id, [])

        for permission in assigned:
            if permission not in expected and permission not in exceptions:
                report.append({
                    "employee_id": employee_id,
                    "employee_name": employee["name"],
                    "role": role,
                    "flagged_permission": permission,
                    "status": "Pending Review",
                    "reason": "Not in standard role permissions or approved exceptions",
                })

    return report

access_review_report = detect_access_drift(
    employees,
    role_permissions,
    assigned_permissions,
    approved_exceptions,
)

print(f"Permissions flagged for review: {len(access_review_report)}")

Permissions flagged for review: 2


## 3. Display the results
Pandas displays the findings as a table. If no permissions are flagged, the table will still show its column headings.

In [8]:
import pandas as pd

columns = [
    "employee_id", "employee_name", "role",
    "flagged_permission", "status", "reason"
]
report_df = pd.DataFrame(access_review_report, columns=columns)
display(report_df)

,employee_id,employee_name,role,flagged_permission,status,reason
0,E002,Meera,HR Executive,financial_reports,Pending Review,Not in standard role permissions or approved e...
1,E002,Meera,HR Executive,customer_database,Pending Review,Not in standard role permissions or approved e...


## 4. Export the report
The CSV is saved in the notebook's current working folder

In [11]:
report_df.to_csv("access_review_report.csv", index=False)

print("Report saved successfully!")

Report saved successfully!


## 5. Try a test


In [12]:
approved_exceptions["E002"] = []

access_review_report = detect_access_drift(
    employees,
    role_permissions,
    assigned_permissions,
    approved_exceptions,
)

report_df = pd.DataFrame(access_review_report, columns=columns)
display(report_df)

,employee_id,employee_name,role,flagged_permission,status,reason
0,E002,Meera,HR Executive,financial_reports,Pending Review,Not in standard role permissions or approved e...
1,E002,Meera,HR Executive,customer_database,Pending Review,Not in standard role permissions or approved e...


## What this prototype demonstrates
- Role-based permission comparison (RBAC concept)
- Handling approved exceptions
- Producing a structured access-review report
- Exporting results to CSV